In [2]:
import marimo as mo
from pathlib import Path
from api.services.gmail import init_gmail_service

client_service_file = 'client_secret.json'
API_SERVICE_NAME = 'gmail'
API_VERSION = 'v1'
SCOPES = ['https://mail.google.com/']
service = init_gmail_service(client_service_file, API_SERVICE_NAME, API_VERSION, SCOPES)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=918720824762-n4l557fr6t3pf9ishcsdogrckbfudq92.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A41835%2F&scope=https%3A%2F%2Fmail.google.com%2F&state=htC6lk08nvoIRm10cEhCGBuEvrDtdI&access_type=offline
gmail v1 service created successfully


### READING EMAILS

In [3]:
from api.services.gmail import get_email_messages, get_email_message_details

In [4]:
messages = get_email_messages(service, max_results=10)

In [6]:
def flatten_message(messages: list):
    for msg in messages:
        details = get_email_message_details(service, msg['id'])
        if details:
            print(f'Subject: {details["subject"]}')
            print(f'From: {details["sender"]}')
            print(f'Recipients: {details["recipients"]}')
            print(f'Body: {details["body"][:50]}...')
            print(f'Snippet: {details["snippet"]}')
            print(f'Has Attachments: {details["has_attachments"]}')
            print(f'Date: {details["date"]}')
            print(f'Star: {details["star"]}')
            print(f'Label: {details["label"]}')
            print("-" * 50)

In [7]:
flatten_message(messages=messages)

Subject: Security alert
From: Google <no-reply@accounts.google.com>
Recipients: fuhadyusuf6@gmail.com
Body: [image: Google]
You allowed mailMaster access to ...
Snippet: You allowed mailMaster access to some of your Google Account data fuhadyusuf6@gmail.com If you didn&#39;t allow mailMaster access to some of your Google Account data, someone else may be trying to
Has Attachments: False
Date: Fri, 18 Jul 2025 07:33:23 GMT
Star: False
Label: UNREAD, CATEGORY_UPDATES, INBOX
--------------------------------------------------
Subject: Security alert
From: Google <no-reply@accounts.google.com>
Recipients: fuhadyusuf6@gmail.com
Body: [image: Google]
You allowed mailMaster access to ...
Snippet: You allowed mailMaster access to some of your Google Account data fuhadyusuf6@gmail.com If you didn&#39;t allow mailMaster access to some of your Google Account data, someone else may be trying to
Has Attachments: False
Date: Thu, 17 Jul 2025 13:52:49 GMT
Star: False
Label: UNREAD, CATEGORY_UPDATES, I

### Sending Emails

In [ ]:
from api.services.gmail import send_email

In [ ]:
to_address = "phurhardeen@gmail.com"
email_subject = "MailMaster testing"
email_body = "This is a sample email sent using the Gmail API"
attachments_dir = Path('./token files')
attachments_files = list(attachments_dir.glob('*'))

In [ ]:
response_email_sent = send_email(
    service,
    to_address,
    email_subject,
    email_body,
    body_type='plain',
    attachment_paths=attachments_files
)
print(response_email_sent)

### Downloading Message Attachments

In [ ]:
from api.services.gmail import download_attachments_all, download_attachments_parent

In [ ]:
user_id = 'me'
msg_id = '198195069913c679'
download_dir = Path('./downloads')

In [ ]:
download_attachments_parent(service, user_id, msg_id, download_dir)

### Search emails

In [ ]:
from api.services.gmail import search_email_conversations, search_emails

In [ ]:
query = "from:me"
email_messages = search_emails(service, query, max_results=10)

In [ ]:
flatten_message(email_messages)

### Manage Gmail Labels

In [ ]:
from api.services.gmail import get_label_details, create_label, list_labels, modify_label, delete_label, map_label_name_to_id

In [ ]:
labels = ['Miscellenous', 'Very Important', 'Shopping', 'Learning', 'Work']
labels_patreon = ['Posts', 'News Letter', 'Promotions']
created_labels = []

In [ ]:
for label in labels:
    try:
        label = create_label(service, label)
        print(f'Label "{label["name"]}" created')
        created_labels.append(label)
        if label['name'] == "Miscellenous":
            patreon_label_id = label['id']
            for sub_label in labels_patreon:
                create_label(service, f"{label['name']}/{sub_label}")
                print(f'Sub-label "{label["name"]}/{sub_label}" created')
    except Exception as e:
        print(f"Label creation failed for '{label}': {e}")

In [ ]:
gmail_label = list_labels(service)

In [ ]:
get_label_details(service, label_id=created_labels[0]['id'])

In [ ]:
modify_label(service, created_labels[0]['id'], name='Misc', color={'textColor': '#eaa032', 'backgroundColor': '#acd123'})

In [ ]:
labels_to_delete = list_labels(service)
for label in labels_to_delete:
    if label['type'] != 'system':
        delete_label(service, label['id'])
        print(f'Deleted {label["name"]}')

### Manage Email Labels

In [ ]:
from api.services.gmail import modify_email_labels

In [ ]:
query = 'from:ALX'
email_messages = search_emails(service, query, max_results=5)

In [ ]:
patreon_label_id = map_label_name_to_id(service, 'Patreon')


In [ ]:
# Adding labels to emails
for email_message in email_messages:
    email_message_detail = get_email_message_details(service, email_message['id'])
    if 'no-reply' in email_message_detail['sender']:
        modify_email_labels(service, 'me', email_message['id'], add_labels=[patreon_label_id, 'STARRED'])
        print(f'Adding "Patreon" label to email: {email_message_detail["subject"]}')

In [ ]:
# Adding labels to emails
for email_message in email_messages:
    email_message_detail = get_email_message_details(service, email_message['id'])
    if 'no-reply' in email_message_detail['sender']:
        modify_email_labels(service, 'me', email_message['id'], remove_labels=[patreon_label_id, 'STARRED'])
        print(f'Removing "Patreon" label to email: {email_message_detail["subject"]}')

### Trashing and Deleting messages

In [ ]:
from api.services.gmail import batch_trash_emails, empty_trash

In [ ]:
target_emails = []
query = 'Subject:Promo'
email_messages = search_emails(service, query, max_results=10)
for email_message in email_messages:
    email_message_detail = get_email_message_details(service, email_message['id'])
    print(f'Email "{email_message_detail["subject"]}" will be moved to trash')
    target_emails.append(email_message['id'])
batch_trash_emails(service, 'me', target_emails)

In [ ]:
empty_trash(service)

### Manage drafts

In [ ]:
from api.services.gmail import send_draft_email, create_draft_email, delete_draft_email, get_draft_email_message_details

In [ ]:
## uses same setup as the normal create email

### Print entire email conversation

In [8]:
from api.services.gmail import get_message_and_replies

In [9]:
email_messages = get_email_messages(service, max_results=1)
message_id = email_messages[0]['id']
msg_chain = get_message_and_replies(service, message_id)
msg_chain

[{'id': '1981c73c0c3cccdb',
  'subject': 'Security alert',
  'from': 'Google <no-reply@accounts.google.com>',
  'date': 'Fri, 18 Jul 2025 07:33:23 GMT',
  'body': "[image: Google]\r\nYou allowed mailMaster access to some of your Google Account data\r\n\r\n\r\nfuhadyusuf6@gmail.com\r\n\r\nIf you didn't allow mailMaster access to some of your Google Account data,\r\nsomeone else may be trying to access your Google Account data.\r\n\r\nTake a moment now to check your account activity and secure your account.\r\nCheck activity\r\n<https://accounts.google.com/AccountChooser?Email=fuhadyusuf6@gmail.com&continue=https://myaccount.google.com/alert/nt/1752824003000?rfn%3D127%26rfnc%3D1%26eid%3D6284156822527781497%26et%3D0>\r\nTo make changes at any time to the access that mailMaster has to your data,\r\ngo to your Google Account\r\n<https://accounts.google.com/AccountChooser?Email=fuhadyusuf6@gmail.com&continue=https://myaccount.google.com/connections/overview/AXgE0HN5FrEUBbsWXyGKIv0BlQjJHvo2t-